In [ ]:
# The official runtime stack first (see OFFICIAL in the builder), from PyTorch's
# cu128 index; torchvision at the version that belongs to that torch, so nothing
# on the image is left half-matched. Then the scorer's own packages.
!pip -q install "torch==2.9.1" "torchaudio==2.9.1" "torchvision==0.24.1" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.0.0" "accelerate>=1.10.1" "safetensors==0.6.2" speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3


In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path


def captured(command):
    # Output is captured and printed - never written straight to the notebook's
    # raw stdout, which is what stalled before.
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


# An earlier kernel's transformers install pulled a torchvision built for
# another torch, and transformers' lazy import of PreTrainedModel died on it.
# The matching torchvision is installed above; if the import still breaks on
# it, torchvision goes - nothing in this run loads a vision model - and the
# removal is printed, not hidden.
probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.9.1+cu128', 'torchaudio': '2.9.1+cu128', 'torchcodec': '0.8.1', 'transformers': '5.0.0', 'accelerate': '>=1.10.1', 'safetensors': '0.6.2', 'numpy': '2.1.0', 'tqdm': '4.67.1', 'einops': '0.8.1', 'scipy': '1.16.2', 'librosa': '0.11.0', 'tiktoken': '0.12.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'safetensors', 'accelerate')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    if want.startswith(">="):
        from packaging.version import Version
        return Version(have.split("+")[0]) >= Version(want[2:])
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {}
for name, want in OFFICIAL.items():
    have = installed(name)
    stack[name] = {"official": want, "installed": have, "matches": matches(have, want),
                   "pinned_here": name in PINNED}
others = ("tokenizers", "huggingface-hub", "soundfile", "speechbrain", "faster-whisper", "ctranslate2",
          "jiwer", "torchvision", "ipykernel", "jupyter-client", "nbclient", "nbconvert", "papermill")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "other_packages": {name: installed(name) for name in others},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:13} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"

gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
for i in range(len(gpus)):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {gpus[i]}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/root').free / 2**30:.0f} GiB free for the HF cache")
# The model (9.1 GB bf16) and the codec (about 4.3 GB bf16) together, plus the
# KV cache, do not fit one 15 GiB T4 with room to spare. The second card holds
# the codec. That is placement; precision is untouched.
assert len(gpus) >= 2, "this recipe places the codec on a second GPU - use T4 x2"

INPUT = Path("/kaggle/input")
found = sorted(INPUT.rglob("repo/scripts/benchmark.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo under /kaggle/input - see the directories above")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
assert (REPO / "scripts/moss15_worker.py").is_file(), "the dataset predates the MOSS v1.5 worker"

os.environ["VOICECLONE_MOSS15_PYTHON"] = sys.executable
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-moss15-v3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Sequential weight materialisation, as in every earlier MOSS run. It changes
# how weights are loaded, not what is computed.
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

# Every stage asks the gates before it starts; every command runs in its own
# process group with its output in its own log. See scripts/kernel_gates.py.
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import (Gates, bench_verdict, budget_verdict, control_verdict,  # noqa: E402
                          report_group, run_bounded, worker_verdict)

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
CAPACITY = {f"cuda:{i}": torch.cuda.get_device_properties(i).total_memory / 2**30
            for i in range(torch.cuda.device_count())}


In [ ]:
# The job that stalled the last probe, without a model: a fake worker through the
# real worker's stdout claim and protocol, 3,000 lines and 20,000 progress
# updates, on this notebook's own display. Nothing is downloaded unless it passes.
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
from huggingface_hub import snapshot_download

if not GATES.skip("download", ("runner",)):
    try:
        start = time.time()
        MODEL_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-TTS-Local-Transformer-v1.5", revision="be7766a6735b98bd793f7c79fb720b4d0f5d13b8"))
        CODEC_DIR = Path(snapshot_download(repo_id="OpenMOSS-Team/MOSS-Audio-Tokenizer-v2", revision="f6e20e543b33d2c252a7ef71bdf8aa71e5ff9169"))
        sizes = {d.name: sum(p.stat().st_size for p in d.rglob("*") if p.is_file()) / 2**30
                 for d in (MODEL_DIR, CODEC_DIR)}
        GATES.record("download", True, f"{(time.time() - start) / 60:.1f} min, " +
                     ", ".join(f"{n} {s:.2f} GiB" for n, s in sizes.items()))
    except Exception as exc:
        GATES.record("download", False, f"{type(exc).__name__}: {exc}")


In [ ]:
driver = Path("/kaggle/working/drive_moss15.py")
driver.write_text("""
import json, subprocess, sys, time

model_dir, codec_dir, reference, text, output, cwd = sys.argv[1:7]
command = [sys.executable, "scripts/moss15_worker.py", "--model-dir", model_dir,
           "--codec-dir", codec_dir, "--device", "cuda", "--codec-device", "auto"]
start_load = time.time()
worker = subprocess.Popen(command, cwd=cwd, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                          text=True, bufsize=1)

def reply():
    while True:
        line = worker.stdout.readline()
        if not line:
            raise SystemExit(f"worker exited with {worker.poll()}")
        try:
            return json.loads(line.strip())
        except json.JSONDecodeError:
            continue

ready = reply()
ready_at = time.time()
print("READY", json.dumps(ready), flush=True)
print(f"LOAD {ready_at - start_load:.1f}s", flush=True)
if not ready.get("ready"):
    raise SystemExit("worker did not load")
job = {"text": text, "reference": reference, "output": output, "seed": 42, "language": "Persian",
       "audio_temperature": 1.7, "audio_top_p": 0.8, "audio_top_k": 25,
       "audio_repetition_penalty": 1.0, "max_new_tokens": 4096}
start = time.time()
worker.stdin.write(json.dumps(job, ensure_ascii=False) + chr(10))
worker.stdin.flush()
answer = reply()
render_seconds = time.time() - start
print(f"RENDER {render_seconds:.1f}s", flush=True)
print("RESULT", json.dumps(answer), flush=True)
worker.stdin.write(json.dumps({"op": "quit"}) + chr(10))
worker.stdin.flush()
worker.wait(timeout=120)
with open("/kaggle/working/moss15_smoke.json", "w") as handle:
    json.dump({"ready": ready, "load_seconds": ready_at - start_load, "result": answer,
               "render_seconds": render_seconds}, handle)
if not answer.get("ok"):
    raise SystemExit("the render failed")
""", encoding="utf-8")

manifest = json.loads((REPO / "eval/corpus/manifest.json").read_text(encoding="utf-8"))
entry = next(e for e in manifest["entries"] if e["speaker"] == "cv02" and e["role"] == "reference")
REFERENCE = REPO / "eval/corpus" / entry["file"]
SENTENCE = "دیروز در کتابخانه شهر، کتابی درباره تاریخ ایران خواندم."
RENDER = Path("/kaggle/working/moss15_worker.wav")
SMOKE = Path("/kaggle/working/moss15_smoke.json")


In [ ]:
FACTS = {}
if not GATES.skip("worker", ("download",)):
    run = run_bounded(
        [sys.executable, "/kaggle/working/drive_moss15.py", str(MODEL_DIR), str(CODEC_DIR),
         str(REFERENCE), SENTENCE, str(RENDER), str(REPO)],
        timeout=2700, monitor_every=120, log=LOGS / "worker.log")
    RUNS["worker"] = run.record()
    print(run.summary())
    print(captured(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"])[1])
    ok, reason, FACTS = worker_verdict(run, RENDER, SMOKE, expect_rate=48000, expect_channels=2)
    GATES.record("worker", ok, reason)
    if FACTS:
        print("load s:", round(FACTS.get("load_seconds", 0), 1),
              "| render s:", round(FACTS.get("render_seconds", 0), 1),
              "| peaks after load:", FACTS.get("ready", {}).get("memory_gib_after_load"),
              "| peaks during render:", FACTS.get("result", {}).get("memory_gib"))


In [ ]:
if not GATES.skip("bench", ("worker",)):
    run = run_bounded(
        [sys.executable, "scripts/benchmark.py", "--label", "moss15-smoke", "--engine", "moss15",
         "--device", "cuda", "--speakers", "cv02", "--sentences", "s1"],
        timeout=3600, cwd=str(REPO), monitor_every=300, log=LOGS / "bench.log")
    RUNS["bench"] = run.record()
    print(run.summary())
    ok, reason = bench_verdict(run, REPO / "eval/results/moss15-smoke.json")
    GATES.record("bench", ok, reason)


In [ ]:
if not GATES.skip("control", ("bench",)):
    run = run_bounded(
        [sys.executable, "scripts/benchmark.py", "--label", "baseline-moss15-env",
         "--score-only", "--replay-from", "baseline", "--device", "cuda"],
        timeout=3600, cwd=str(REPO), monitor_every=300, log=LOGS / "control.log")
    RUNS["control"] = run.record()
    print(run.summary())
    check = None
    if run.ok and (REPO / "eval/results/baseline-moss15-env.json").is_file():
        import compare_runs

        results = REPO / "eval/results"
        check = compare_runs.scale_control(compare_runs.load("baseline", results),
                                           compare_runs.load("baseline-moss15-env", results))
    ok, reason = control_verdict(run, check)
    GATES.record("control", ok, reason)


In [ ]:
# The full run is priced from what the smoke run measured, not guessed: the
# worker's load time, one render's generation time plus a scoring allowance,
# and the peak memory each card reached, against the time budget and each
# card's capacity.
if not GATES.skip("budget", ("worker", "bench", "control")):
    peaks = {}
    for snapshot in (FACTS.get("ready", {}).get("memory_gib_after_load", {}),
                     FACTS.get("result", {}).get("memory_gib", {})):
        for card, stats in snapshot.items():
            peaks[card] = max(peaks.get(card, 0.0), stats.get("peak_reserved", 0.0))
    ok, reason = budget_verdict(load_seconds=FACTS.get("load_seconds", 0.0),
                                cell_seconds=FACTS["result"]["generate_seconds"] + 20,
                                cells=27, timeout=14400, peaks_gib=peaks, capacity_gib=CAPACITY)
    GATES.record("budget", ok, reason)

if not GATES.skip("full", ("budget",)):
    run = run_bounded(
        [sys.executable, "scripts/benchmark.py", "--label", "moss15", "--engine", "moss15", "--device", "cuda"],
        timeout=14400, cwd=str(REPO), monitor_every=600, log=LOGS / "full.log")
    RUNS["full"] = run.record()
    print(run.summary())
    path = REPO / "eval/results/moss15.json"
    summary = json.loads(path.read_text())["summary"] if path.is_file() else {}
    complete = (run.ok and summary.get("renders") == 27
                and not summary.get("failures") and not summary.get("untranscribed"))
    GATES.record("full", complete, json.dumps(summary) if summary else f"{run.outcome}, no results file")


In [ ]:
destination = Path("/kaggle/working/results")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)
native = REPO / "outputs/native/moss15"
if native.is_dir():
    shutil.copytree(native, Path("/kaggle/working/moss15_native"), dirs_exist_ok=True)
renders = Path("/kaggle/working/moss15_renders")
renders.mkdir(exist_ok=True)
for folder in ("moss15-smoke", "moss15"):
    for path in sorted((REPO / "outputs/benchmark" / folder).glob("*.wav")):
        shutil.copy2(path, renders / f"{folder}__{path.name}")
if RENDER.is_file():
    shutil.copy2(RENDER, renders / RENDER.name)
for extra in (SMOKE, Path("/kaggle/working/environment.json")):
    if extra.is_file():
        shutil.copy2(extra, destination / extra.name)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
print("gates:", json.dumps(GATES.summary(), indent=1))
print("saved:", len(list(renders.glob("*.wav"))), "harness renders,",
      len(list(Path("/kaggle/working/moss15_native").glob("*.wav"))) if native.is_dir() else 0, "native")
